# Support Vector Machine (SVM) - Complete Guide
## Breast Cancer Classification using SVM

**Created by:** Mahipal Singh, Sr. Instructor and Mentor - DS/AI/ML

---

### Notebook Objective:
This notebook provides an end-to-end guide to understanding and implementing Support Vector Machines (SVM) using the Breast Cancer dataset. We'll cover:
- What is SVM and how it works
- Data loading and exploration
- Data preprocessing and feature scaling
- SVM model training
- Model evaluation and performance metrics
- Hyperparameter tuning
- Visualization of results

Let's dive into the world of Support Vector Machines! 🚀

---
## Section 1: Introduction to Support Vector Machine (SVM)

### What is SVM?
Support Vector Machine is a supervised machine learning algorithm that can be used for classification and regression problems. It's particularly effective for binary classification but can be extended to multi-class problems.

### Key Concepts:
1. **Hyperplane**: A decision boundary that separates different classes in the feature space
2. **Support Vectors**: Data points that lie closest to the hyperplane and are critical in determining its position
3. **Margin**: The distance between the hyperplane and the support vectors. SVM aims to maximize this margin
4. **Kernel Trick**: A technique to handle non-linearly separable data by transforming it into a higher-dimensional space

### How SVM Works:
- SVM finds the optimal hyperplane that maximizes the margin between two classes
- Points on the margin are called support vectors
- For non-linear problems, kernels transform the data into higher dimensions where linear separation is possible

### Advantages of SVM:
✓ Effective in high-dimensional spaces
✓ Works well with non-linear data (using kernel trick)
✓ Memory efficient (uses only support vectors)
✓ Versatile kernel functions
✓ Great for binary and multi-class classification

### Disadvantages of SVM:
✗ Slow with large datasets
✗ Requires feature scaling
✗ Difficult to interpret predictions
✗ Choosing the right kernel can be challenging

---
## Section 2: Import Libraries and Load Data

In [ ]:
# ============================================================================
# STEP 1: Import Required Libraries
# ============================================================================
# Standard libraries for data manipulation and numerical operations
import numpy as np                          # Numerical computing
import pandas as pd                         # Data manipulation and analysis
import warnings
warnings.filterwarnings('ignore')           # Suppress warning messages

# Visualization libraries
import matplotlib.pyplot as plt             # Plotting library
import seaborn as sns                       # Statistical data visualization
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)   # Set default figure size

# Machine Learning libraries
from sklearn.datasets import load_breast_cancer              # Load breast cancer dataset
from sklearn.model_selection import train_test_split         # Split data into train/test
from sklearn.preprocessing import StandardScaler              # Standardize features
from sklearn.svm import SVC                                  # Support Vector Classifier
from sklearn.model_selection import GridSearchCV             # Hyperparameter tuning
from sklearn.model_selection import cross_val_score          # Cross-validation

# Evaluation metrics
from sklearn.metrics import (classification_report, 
                              confusion_matrix, 
                              accuracy_score,
                              precision_score,
                              recall_score,
                              f1_score,
                              roc_auc_score,
                              roc_curve)

print("✓ All libraries imported successfully!")

In [ ]:
# ============================================================================
# STEP 2: Load the Breast Cancer Dataset
# ============================================================================
# The breast cancer dataset contains measurements of tumor characteristics
# and whether the tumor is malignant (cancerous) or benign (non-cancerous)

# Load the dataset from sklearn
data = load_breast_cancer()

# Extract features (X) and target variable (y)
X = data.data                               # Feature matrix (30 features)
y = data.target                             # Target variable (0: Malignant, 1: Benign)
feature_names = data.feature_names          # Names of the 30 features

# Create a DataFrame for better visualization
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Display basic information
print("="*80)
print("BREAST CANCER DATASET INFORMATION")
print("="*80)
print(f"Dataset Shape: {df.shape}")
print(f"Number of Samples: {df.shape[0]}")
print(f"Number of Features: {df.shape[1] - 1}")
print(f"\nTarget Variable (y) - Class Distribution:")
print(f"  - Malignant (0): {(y == 0).sum()} samples ({(y == 0).sum()/len(y)*100:.2f}%)")
print(f"  - Benign (1):    {(y == 1).sum()} samples ({(y == 1).sum()/len(y)*100:.2f}%)")
print(f"\nFirst few rows of the dataset:")
print(df.head())

---
## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================================
# STEP 3: Exploratory Data Analysis - Understand the Data
# ============================================================================

# Check for missing values
print("\n" + "="*80)
print("MISSING VALUES CHECK")
print("="*80)
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("✓ No missing values found in the dataset!")
else:
    print(missing_values)

# Statistical summary
print("\n" + "="*80)
print("STATISTICAL SUMMARY")
print("="*80)
print(df.describe())

In [ ]:
# ============================================================================
# STEP 4: Visualize Feature Distributions
# ============================================================================

# Create visualizations to understand the data better
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Class Distribution (Bar Plot)
ax1 = axes[0, 0]
class_counts = df['target'].value_counts()
colors = ['#FF6B6B', '#4ECDC4']
ax1.bar(['Malignant (0)', 'Benign (1)'], class_counts.values, color=colors)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Distribution of Target Classes', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Distribution of Mean Radius (first feature)
ax2 = axes[0, 1]
ax2.hist(df[df['target'] == 0]['mean radius'], bins=20, alpha=0.6, label='Malignant', color='#FF6B6B')
ax2.hist(df[df['target'] == 1]['mean radius'], bins=20, alpha=0.6, label='Benign', color='#4ECDC4')
ax2.set_xlabel('Mean Radius', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Mean Radius by Class', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Plot 3: Distribution of Mean Texture
ax3 = axes[1, 0]
ax3.hist(df[df['target'] == 0]['mean texture'], bins=20, alpha=0.6, label='Malignant', color='#FF6B6B')
ax3.hist(df[df['target'] == 1]['mean texture'], bins=20, alpha=0.6, label='Benign', color='#4ECDC4')
ax3.set_xlabel('Mean Texture', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('Distribution of Mean Texture by Class', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Plot 4: Pie Chart
ax4 = axes[1, 1]
ax4.pie(class_counts.values, labels=['Malignant', 'Benign'], autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 11})
ax4.set_title('Percentage Distribution of Classes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")

---
## Section 4: Data Preprocessing and Feature Scaling

### Why is Feature Scaling Important for SVM?
SVM is a **distance-based algorithm** that computes distances between data points in the feature space.
If features have different scales (ranges), features with larger values will dominate the distance calculation.

**Example:**
- Feature 1 ranges from 0-1000
- Feature 2 ranges from 0-1
- Feature 1 will dominate the distance calculation

**Solution: StandardScaler (Standardization)**
- Transforms features to have mean=0 and standard deviation=1
- Formula: Z = (X - mean) / std_dev
- Makes all features equally important

In [ ]:
# ============================================================================
# STEP 5: Split Data into Training and Testing Sets
# ============================================================================
# We split the data to:
# - Training set (70%): Used to train the SVM model
# - Testing set (30%):  Used to evaluate model performance
# This helps us evaluate how well the model generalizes to unseen data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,          # 30% for testing, 70% for training
    random_state=42,        # Fixed seed for reproducibility
    stratify=y              # Maintain class distribution in both sets
)

print("="*80)
print("DATA SPLITTING RESULTS")
print("="*80)
print(f"Training Set Size: {X_train.shape[0]} samples")
print(f"Testing Set Size:  {X_test.shape[0]} samples")
print(f"Training Set Class Distribution:")
print(f"  - Malignant (0): {(y_train == 0).sum()}")
print(f"  - Benign (1):    {(y_train == 1).sum()}")
print(f"Testing Set Class Distribution:")
print(f"  - Malignant (0): {(y_test == 0).sum()}")
print(f"  - Benign (1):    {(y_test == 1).sum()}")

In [ ]:
# ============================================================================
# STEP 6: Feature Scaling / Standardization
# ============================================================================
# StandardScaler standardizes features by removing the mean and scaling to unit variance
# Formula: Z = (X - mean) / standard_deviation
# This is CRUCIAL for SVM because it's a distance-based algorithm

# Create a StandardScaler object
scaler = StandardScaler()

# Fit the scaler on training data and transform both train and test data
# IMPORTANT: We fit on training data only to avoid data leakage!
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train data
X_test_scaled = scaler.transform(X_test)       # Only transform test data

print("\n" + "="*80)
print("FEATURE SCALING RESULTS")
print("="*80)
print("\nBefore Scaling:")
print(f"  Mean of first feature: {X_train[:, 0].mean():.2f}")
print(f"  Std  of first feature: {X_train[:, 0].std():.2f}")
print(f"  Range: [{X_train[:, 0].min():.2f}, {X_train[:, 0].max():.2f}]")

print("\nAfter Scaling:")
print(f"  Mean of first feature: {X_train_scaled[:, 0].mean():.6f} (≈ 0)")
print(f"  Std  of first feature: {X_train_scaled[:, 0].std():.6f} (≈ 1)")
print(f"  Range: [{X_train_scaled[:, 0].min():.2f}, {X_train_scaled[:, 0].max():.2f}]")

print("\n✓ Feature scaling completed!")

---
## Section 5: Train SVM Model

### SVM Kernel Types:
1. **Linear Kernel**: Best for linearly separable data
2. **Polynomial Kernel**: Good for data with polynomial relationships
3. **RBF (Radial Basis Function)**: Best for most cases, can handle non-linear data
4. **Sigmoid Kernel**: Similar to neural network activation

### Important Parameters:
- **C (Regularization Parameter)**: Controls the trade-off between maximizing margin and minimizing classification error
  - Small C: Large margin, more misclassifications (underfitting)
  - Large C: Small margin, fewer misclassifications (overfitting)
- **gamma (for RBF kernel)**: Defines how far the influence of a single training example reaches
  - Small gamma: Each point has far-reaching influence (smooth decision boundary)
  - Large gamma: Each point has close influence (complex decision boundary)

In [ ]:
# ============================================================================
# STEP 7: Train SVM Model with RBF Kernel (Default)
# ============================================================================
# RBF (Radial Basis Function) kernel is most versatile and works well for most problems

# Create and train the SVM model
svm_model = SVC(
    kernel='rbf',           # Use RBF kernel (good for non-linear problems)
    C=1.0,                  # Regularization parameter (default)
    gamma='scale',          # Kernel coefficient (automatic scaling)
    probability=True,       # Enable probability estimates
    random_state=42         # For reproducibility
)

# Train the model on scaled training data
print("Training SVM Model (RBF Kernel)...")
svm_model.fit(X_train_scaled, y_train)
print("✓ Model training completed!")

# Display model information
print("\n" + "="*80)
print("SVM MODEL INFORMATION")
print("="*80)
print(f"Number of Support Vectors: {len(svm_model.support_vectors_)}")
print(f"Percentage of Support Vectors: {len(svm_model.support_vectors_)/len(X_train)*100:.2f}%")
print(f"Support Vector Indices (first 10): {svm_model.support_[:10]}")

In [ ]:
# ============================================================================
# STEP 8: Make Predictions
# ============================================================================
# Using the trained model to predict on both training and testing data

# Predictions on training data
y_train_pred = svm_model.predict(X_train_scaled)

# Predictions on testing data
y_test_pred = svm_model.predict(X_test_scaled)

# Probability predictions (useful for ROC-AUC score)
y_test_pred_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print("✓ Predictions completed!")
print(f"\nFirst 10 predictions on test set: {y_test_pred[:10]}")
print(f"First 10 actual values on test set: {y_test[:10]}")

---
## Section 6: Model Evaluation

### Evaluation Metrics Explained:

**Confusion Matrix:**
```
                 Predicted
                 Neg  Pos
Actual  Neg  |  TN  | FP  |
        Pos  |  FN  | TP  |
```
- **TN (True Negative)**: Correctly predicted negative
- **FP (False Positive)**: Incorrectly predicted positive (Type I error)
- **FN (False Negative)**: Incorrectly predicted negative (Type II error) - **More critical in medical diagnosis!**
- **TP (True Positive)**: Correctly predicted positive

**Accuracy**: (TP + TN) / Total - Overall correctness

**Precision**: TP / (TP + FP) - Among positive predictions, how many are actually positive?

**Recall (Sensitivity)**: TP / (TP + FN) - Among actual positives, how many did we catch? **Important in medical diagnosis!**

**F1-Score**: Harmonic mean of Precision and Recall - Balances both metrics

**ROC-AUC**: Area under the Receiver Operating Characteristic curve - Measures model's ability to distinguish between classes

In [ ]:
# ============================================================================
# STEP 9: Calculate Evaluation Metrics
# ============================================================================

# Calculate accuracy scores
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Calculate other metrics
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_pred_proba)

# Print results
print("="*80)
print("MODEL PERFORMANCE METRICS")
print("="*80)
print(f"\nTraining Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Testing Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"\nPrecision: {precision:.4f} (Of predicted positive, {precision*100:.2f}% are actually positive)")
print(f"Recall:    {recall:.4f} (Of actual positive, we caught {recall*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} (Harmonic mean of Precision and Recall)")
print(f"ROC-AUC:   {roc_auc:.4f} (Model's ability to distinguish between classes)")

# Check for overfitting
print(f"\nOverfitting Analysis:")
print(f"Difference (Train - Test Accuracy): {train_accuracy - test_accuracy:.4f}")
if train_accuracy - test_accuracy > 0.05:
    print("⚠ Model shows signs of overfitting!")
else:
    print("✓ Model generalization looks good!")

In [ ]:
# ============================================================================
# STEP 10: Confusion Matrix Analysis
# ============================================================================

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Calculate metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*80)
print("CONFUSION MATRIX ANALYSIS")
print("="*80)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nTrue Negatives (TN):  {tn}  - Correctly predicted as Malignant")
print(f"False Positives (FP): {fp}  - Incorrectly predicted as Benign (missed cancer)")
print(f"False Negatives (FN): {fn}  - Incorrectly predicted as Malignant (false alarm)")
print(f"True Positives (TP):  {tp}  - Correctly predicted as Benign")

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            cbar_kws={'label': 'Count'})
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.title('Confusion Matrix - SVM Breast Cancer Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# STEP 11: Classification Report
# ============================================================================

print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred,
                          target_names=['Malignant', 'Benign'],
                          digits=4))

In [ ]:
# ============================================================================
# STEP 12: ROC Curve Visualization
# ============================================================================
# ROC (Receiver Operating Characteristic) curve shows the trade-off between
# True Positive Rate and False Positive Rate at different classification thresholds

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_pred_proba)
roc_auc_score_val = roc_auc_score(y_test, y_test_pred_proba)

# Plot ROC curve
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color='#1f77b4', lw=2, label=f'ROC Curve (AUC = {roc_auc_score_val:.4f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve - SVM Breast Cancer Classification', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ ROC curve plotted!")
print(f"\nROC-AUC Score: {roc_auc_score_val:.4f}")
print("AUC Interpretation:")
print("  - 0.5: Random guessing")
print("  - 0.7-0.8: Acceptable")
print("  - 0.8-0.9: Excellent")
print("  - > 0.9: Outstanding")

---
## Section 7: Hyperparameter Tuning with GridSearchCV

### What is Hyperparameter Tuning?
Hyperparameter tuning involves finding the best combination of parameters that maximize model performance.

### GridSearchCV:
- Exhaustively searches through a specified parameter grid
- Uses cross-validation to evaluate each combination
- Finds the best parameters that generalize well to unseen data

### Parameters to Tune:
- **C**: Regularization strength (0.01 to 100)
- **gamma**: Kernel coefficient for RBF (0.0001 to 1)
- **kernel**: Type of kernel (linear, rbf, poly, sigmoid)

In [ ]:
# ============================================================================
# STEP 13: Hyperparameter Tuning using GridSearchCV
# ============================================================================
# This step finds the optimal parameters for our SVM model
# WARNING: This may take a few minutes to complete

print("Starting Hyperparameter Tuning...")
print("This may take a minute or two...\n")

# Define parameter grid to search
param_grid = {
    'C': [0.1, 1, 10, 100],           # Regularization parameter
    'gamma': [0.0001, 0.001, 0.01, 0.1, 'scale'],  # Kernel coefficient
    'kernel': ['rbf', 'linear']       # Kernel type
}

# Create GridSearchCV object
# cv=5 means 5-fold cross-validation
# n_jobs=-1 uses all available processors
grid_search = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=5,                             # 5-fold cross-validation
    n_jobs=-1,                        # Use all processors
    verbose=1                         # Print progress
)

# Fit GridSearchCV
grid_search.fit(X_train_scaled, y_train)

print("\n" + "="*80)
print("HYPERPARAMETER TUNING RESULTS")
print("="*80)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

# Get the best model
best_svm = grid_search.best_estimator_

In [ ]:
# ============================================================================
# STEP 14: Evaluate Best Model
# ============================================================================

# Make predictions with the best model
y_test_pred_best = best_svm.predict(X_test_scaled)
y_test_pred_proba_best = best_svm.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
best_accuracy = accuracy_score(y_test, y_test_pred_best)
best_precision = precision_score(y_test, y_test_pred_best)
best_recall = recall_score(y_test, y_test_pred_best)
best_f1 = f1_score(y_test, y_test_pred_best)
best_roc_auc = roc_auc_score(y_test, y_test_pred_proba_best)

print("\n" + "="*80)
print("BEST MODEL PERFORMANCE")
print("="*80)
print(f"\nAccuracy:   {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"Precision:  {best_precision:.4f}")
print(f"Recall:     {best_recall:.4f}")
print(f"F1-Score:   {best_f1:.4f}")
print(f"ROC-AUC:    {best_roc_auc:.4f}")

# Compare with initial model
print(f"\n" + "="*80)
print("COMPARISON: Initial vs Optimized Model")
print("="*80)
print(f"\nMetric          Initial Model    Optimized Model   Improvement")
print("-" * 70)
print(f"Accuracy        {test_accuracy:.4f}            {best_accuracy:.4f}            {best_accuracy-test_accuracy:+.4f}")
print(f"Precision       {precision:.4f}            {best_precision:.4f}            {best_precision-precision:+.4f}")
print(f"Recall          {recall:.4f}            {best_recall:.4f}            {best_recall-recall:+.4f}")
print(f"F1-Score        {f1:.4f}            {best_f1:.4f}            {best_f1-f1:+.4f}")
print(f"ROC-AUC         {roc_auc:.4f}            {best_roc_auc:.4f}            {best_roc_auc-roc_auc:+.4f}")

---
## Section 8: Cross-Validation Analysis

In [ ]:
# ============================================================================
# STEP 15: Cross-Validation Analysis
# ============================================================================
# Cross-validation helps us understand model stability and generalization
# k-Fold Cross-Validation divides data into k subsets and trains k models

# Perform 5-fold cross-validation
cv_scores = cross_val_score(best_svm, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("\n" + "="*80)
print("CROSS-VALIDATION ANALYSIS (5-Fold)")
print("="*80)
print(f"\nCross-Validation Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")
print(f"\nConfidence Interval (±1 std): [{cv_scores.mean()-cv_scores.std():.4f}, {cv_scores.mean()+cv_scores.std():.4f}]")

# Visualize cross-validation scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: CV Scores for each fold
folds = np.arange(1, len(cv_scores) + 1)
ax1.bar(folds, cv_scores, color='#4ECDC4', alpha=0.7, edgecolor='black')
ax1.axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax1.set_xlabel('Fold Number', fontsize=12)
ax1.set_ylabel('Accuracy Score', fontsize=12)
ax1.set_title('Cross-Validation Scores by Fold', fontsize=14, fontweight='bold')
ax1.set_xticks(folds)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0.8, 1.0])

# Plot 2: Distribution of CV Scores
ax2.hist(cv_scores, bins=5, color='#FF6B6B', alpha=0.7, edgecolor='black')
ax2.axvline(x=cv_scores.mean(), color='darkred', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax2.set_xlabel('Accuracy Score', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Cross-Validation Scores', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Model shows consistent performance across different data splits!")

---
## Section 9: Feature Importance Analysis

In [ ]:
# ============================================================================
# STEP 16: Feature Importance Analysis
# ============================================================================
# While SVM doesn't directly provide feature importance, we can analyze
# the coefficients of the support vectors and their distances from hyperplane

# For linear kernel SVM, we can look at the coefficient magnitudes
# But since we're using RBF kernel, let's use permutation importance

from sklearn.inspection import permutation_importance

# Calculate permutation importance
perm_importance = permutation_importance(
    best_svm, X_test_scaled, y_test,
    n_repeats=10,
    random_state=42
)

# Create a dataframe for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': perm_importance.importances_mean
}).sort_values('Importance', ascending=False)

# Display top 10 features
print("\n" + "="*80)
print("TOP 10 IMPORTANT FEATURES")
print("="*80)
print(importance_df.head(10).to_string(index=False))

# Visualize top features
plt.figure(figsize=(12, 6))
top_features = importance_df.head(10)
plt.barh(range(len(top_features)), top_features['Importance'], color='#1f77b4', alpha=0.7)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Permutation Importance', fontsize=12)
plt.title('Top 10 Most Important Features for Classification', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 10: Summary and Conclusions

In [ ]:
# ============================================================================
# STEP 17: Final Summary and Conclusions
# ============================================================================

print("\n" + "="*80)
print("FINAL MODEL SUMMARY AND CONCLUSIONS")
print("="*80)

print("\n📊 MODEL PERFORMANCE:")
print("-" * 80)
print(f"Accuracy:   {best_accuracy*100:.2f}%")
print(f"Precision:  {best_precision*100:.2f}%")
print(f"Recall:     {best_recall*100:.2f}%")
print(f"F1-Score:   {best_f1:.4f}")
print(f"ROC-AUC:    {best_roc_auc:.4f}")
print(f"CV Score:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\n🎯 OPTIMAL HYPERPARAMETERS:")
print("-" * 80)
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

print("\n✅ KEY ACHIEVEMENTS:")
print("-" * 80)
print(f"✓ Successfully classified breast cancer cases with {best_accuracy*100:.2f}% accuracy")
print(f"✓ Achieved high recall ({best_recall*100:.2f}%), important for medical diagnosis")
print(f"✓ Model is well-generalized (CV score matches test score)")
print(f"✓ Identified {len(best_svm.support_vectors_)} support vectors")
print(f"✓ ROC-AUC score of {best_roc_auc:.4f} indicates excellent discrimination ability")

print("\n📈 INSIGHTS:")
print("-" * 80)
print(f"• {len(best_svm.support_vectors_)} out of {len(X_train)} training samples are support vectors")
print(f"• Top important feature: {importance_df.iloc[0]['Feature']}")
print(f"• Model shows {'overfitting' if train_accuracy - test_accuracy > 0.05 else 'good generalization'}")
print(f"• Class balance: {(y==0).sum()/(y==1).sum():.2f} malignant per benign case")

print("\n💡 PRACTICAL RECOMMENDATIONS:")
print("-" * 80)
print("1. This model can assist radiologists in breast cancer diagnosis")
print("2. High recall (sensitivity) means fewer false negatives - critical for medical use")
print("3. Consider ensemble methods for even better performance")
print("4. Regular model retraining with new data is recommended")
print("5. Combine with domain expertise for clinical decision-making")

print("\n" + "="*80)
print("Notebook completed successfully! 🎉")
print("="*80)

---
## Additional Resources and References

### Key Concepts Review:
1. **Support Vector Machine (SVM)**: A powerful algorithm for classification that finds the optimal hyperplane
2. **Kernel Trick**: Transforms data to handle non-linear separability
3. **Support Vectors**: Critical data points that define the decision boundary
4. **Feature Scaling**: Essential preprocessing step for distance-based algorithms
5. **Cross-Validation**: Technique to assess model generalization

### Python Libraries Used:
- **scikit-learn**: Machine learning library with SVM implementation
- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computing
- **matplotlib & seaborn**: Data visualization

### Further Learning:
- Explore different kernels: polynomial, sigmoid
- Try multi-class classification (One-vs-Rest, One-vs-One)
- Implement SVM from scratch using optimization algorithms
- Compare SVM with other algorithms (Random Forest, Gradient Boosting, etc.)
- Learn about Kernel methods and support vector regression

### Medical Context:
- The breast cancer dataset contains features computed from digitized images
- Features include: radius, texture, perimeter, area, smoothness, etc.
- Class 0 (Malignant): Cancerous tumors
- Class 1 (Benign): Non-cancerous tumors

---

**Created by:** Mahipal Singh, Sr. Instructor and Mentor - DS/AI/ML

**Last Updated:** 2026

**Disclaimer:** This notebook is for educational purposes. For medical diagnosis, always consult with healthcare professionals.

In [ ]:
# ============================================================================
# BONUS: Quick Reference Guide
# ============================================================================

print("\n" + "="*80)
print("QUICK REFERENCE GUIDE - SVM KEY POINTS")
print("="*80)

quick_ref = """
🔹 SVM BASICS:
   • Supervised learning algorithm for classification
   • Finds optimal hyperplane with maximum margin
   • Works well in high-dimensional spaces

🔹 KERNEL TYPES:
   • Linear: Fast, for linearly separable data
   • RBF: Default choice, handles non-linear data
   • Polynomial: For polynomial relationships
   • Sigmoid: Similar to neural networks

🔹 HYPERPARAMETERS:
   • C: Regularization (small=high margin, large=low margin)
   • gamma: Kernel coefficient (small=smooth, large=complex)
   • kernel: Choice of kernel function

🔹 PREPROCESSING:
   • Feature scaling is CRITICAL (StandardScaler)
   • Handle missing values
   • Imbalanced classes (use class_weight parameter)

🔹 EVALUATION METRICS:
   • Accuracy: Overall correctness
   • Precision: True positives among predicted positives
   • Recall: True positives among actual positives
   • F1-Score: Harmonic mean of precision and recall
   • ROC-AUC: Area under receiver operating characteristic curve

🔹 TUNING STRATEGY:
   1. Start with RBF kernel and default parameters
   2. Use GridSearchCV for hyperparameter tuning
   3. Use cross-validation for robust evaluation
   4. Check for overfitting/underfitting
   5. Fine-tune based on business requirements

🔹 COMPLEXITY ANALYSIS:
   • Training time: O(n²) to O(n³) where n = number of samples
   • Prediction time: O(n_support_vectors × n_features)
   • Space complexity: O(n_support_vectors)
"""

print(quick_ref)

In [ ]:
# ============================================================================
# Practice Exercise for Students
# ============================================================================

print("\n" + "="*80)
print("PRACTICE EXERCISES FOR STUDENTS")
print("="*80)

exercises = """
🎓 BEGINNER LEVEL:

1. Try different kernels and compare their performance:
   - Change kernel parameter to 'linear', 'poly', 'rbf', 'sigmoid'
   - Record accuracy for each kernel
   - Which kernel performs best?

2. Experiment with different train-test splits:
   - Try: 60-40, 70-30, 80-20, 90-10
   - How does it affect model performance?

---

🎓 INTERMEDIATE LEVEL:

3. Feature Selection:
   - Select only top 5 features based on importance
   - Train model with reduced features
   - Compare accuracy and training time

4. Class Imbalance Handling:
   - Use class_weight='balanced' parameter
   - Compare with current model
   - How does it affect recall and precision?

5. Probability Threshold Tuning:
   - Change decision threshold from 0.5 to other values
   - Plot precision-recall tradeoff

---

🎓 ADVANCED LEVEL:

6. Ensemble Methods:
   - Combine SVM with other classifiers (Random Forest, XGBoost)
   - Use VotingClassifier or StackingClassifier
   - Compare with single SVM model

7. Custom Kernel:
   - Implement custom kernel function
   - Compare with built-in kernels

8. Learning Curve Analysis:
   - Plot learning curves (training and validation accuracy vs. dataset size)
   - Diagnose bias-variance tradeoff

---

💡 DISCUSSION QUESTIONS:

• Why is feature scaling important for SVM?
• How does changing C parameter affect the model?
• When would you choose SVM over other algorithms?
• How would you handle a severely imbalanced dataset?
• What are the limitations of SVM with large datasets?
"""

print(exercises)
print("\n" + "="*80)